In [4]:
import cv2
import time
import math as m
import mediapipe as mp


# ================= UTILS =================
def findDistance(x1, y1, x2, y2):
    return m.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


def findAngle(x1, y1, x2, y2):
    dx = x2 - x1
    dy = y2 - y1
    angle = m.degrees(m.atan2(dy, dx))
    return abs(angle)


def sendWarning():
    print("⚠️ Bad posture detected!")


# ================= INIT =================
good_frames = 0
bad_frames = 0

font = cv2.FONT_HERSHEY_SIMPLEX

green = (127, 255, 0)
red = (50, 50, 255)
yellow = (0, 255, 255)
pink = (255, 0, 255)
light_green = (127, 233, 100)

# Mediapipe setup
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5,
                    min_tracking_confidence=0.5)

# ================= VIDEO =================
file_name = "input.mp4"   # OR use 0 for webcam
cap = cv2.VideoCapture(file_name)

# fallback to webcam
if not cap.isOpened():
    print("Switching to webcam...")
    cap = cv2.VideoCapture(0)

fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:
    fps = 30  # fallback

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_output = cv2.VideoWriter('output.mp4', fourcc, fps, (width, height))


# ================= MAIN LOOP =================
print("Processing... Press 'q' to quit")

while cap.isOpened():
    success, image = cap.read()
    if not success:
        break

    h, w = image.shape[:2]

    # Convert to RGB
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)

    # 🚨 IMPORTANT FIX: avoid crash
    if results.pose_landmarks is None:
        cv2.imshow("Posture Detection", image)
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break
        continue

    lm = results.pose_landmarks
    lmPose = mp_pose.PoseLandmark

    # ================= LANDMARKS =================
    l_shldr = lm.landmark[lmPose.LEFT_SHOULDER]
    r_shldr = lm.landmark[lmPose.RIGHT_SHOULDER]
    l_ear = lm.landmark[lmPose.LEFT_EAR]
    l_hip = lm.landmark[lmPose.LEFT_HIP]

    l_shldr_x, l_shldr_y = int(l_shldr.x * w), int(l_shldr.y * h)
    r_shldr_x, r_shldr_y = int(r_shldr.x * w), int(r_shldr.y * h)
    l_ear_x, l_ear_y = int(l_ear.x * w), int(l_ear.y * h)
    l_hip_x, l_hip_y = int(l_hip.x * w), int(l_hip.y * h)

    # ================= ALIGNMENT =================
    offset = findDistance(l_shldr_x, l_shldr_y, r_shldr_x, r_shldr_y)

    if offset < 100:
        cv2.putText(image, "Aligned", (w - 150, 30), font, 0.8, green, 2)
    else:
        cv2.putText(image, "Not Aligned", (w - 150, 30), font, 0.8, red, 2)

    # ================= ANGLES =================
    neck = findAngle(l_shldr_x, l_shldr_y, l_ear_x, l_ear_y)
    torso = findAngle(l_hip_x, l_hip_y, l_shldr_x, l_shldr_y)

    text = f"Neck: {int(neck)}  Torso: {int(torso)}"

    # ================= POSTURE =================
    if neck < 40 and torso < 10:
        good_frames += 1
        bad_frames = 0
        color = light_green
    else:
        bad_frames += 1
        good_frames = 0
        color = red

    cv2.putText(image, text, (10, 30), font, 0.8, color, 2)

    # ================= TIME =================
    good_time = good_frames / fps
    bad_time = bad_frames / fps

    if good_time > 0:
        cv2.putText(image, f"Good: {round(good_time,1)}s",
                    (10, h - 20), font, 0.8, green, 2)
    else:
        cv2.putText(image, f"Bad: {round(bad_time,1)}s",
                    (10, h - 20), font, 0.8, red, 2)

    # ================= ALERT =================
    if bad_time > 10:   # change to 180 for real use
        sendWarning()

    # ================= DRAW =================
    cv2.circle(image, (l_shldr_x, l_shldr_y), 6, yellow, -1)
    cv2.circle(image, (l_ear_x, l_ear_y), 6, yellow, -1)
    cv2.circle(image, (l_hip_x, l_hip_y), 6, yellow, -1)
    cv2.circle(image, (r_shldr_x, r_shldr_y), 6, pink, -1)

    cv2.line(image, (l_shldr_x, l_shldr_y), (l_ear_x, l_ear_y), color, 3)
    cv2.line(image, (l_hip_x, l_hip_y), (l_shldr_x, l_shldr_y), color, 3)

    # Save video
    video_output.write(image)

    cv2.imshow("Posture Detection", image)

    if cv2.waitKey(5) & 0xFF == ord('q'):
        break


# ================= CLEANUP =================
cap.release()
video_output.release()
cv2.destroyAllWindows()
print("Finished")

AttributeError: module 'mediapipe' has no attribute 'solutions'